In [29]:
from IPython.display import clear_output
clear_output()

In [30]:
import sys
sys.path.insert(0, '/storage/work/fxs5261/venv/lib/python3.x/site-packages')
print(sys.executable)

import os
os.environ["HF_HOME"] = "/storage/work/fxs5261/huggingface_cache"

/storage/icds/RISE/sw8/anaconda/anaconda3/bin/python


In [45]:
!/storage/icds/RISE/sw8/anaconda/anaconda3/bin/python -m pip install bertopic sentence-transformers detoxify

import sys
sys.path.insert(0, '/storage/work/fxs5261/.local/lib/python3.9/site-packages')

import pandas as pd
import os
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from bertopic import BERTopic

def simple_clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [w for w in text.split() if w not in ENGLISH_STOP_WORDS and len(w) > 2]
    return " ".join(tokens)

Defaulting to user installation because normal site-packages is not writeable


ImportError: cannot import name 'HDBSCAN' from 'sklearn.cluster' (/storage/icds/RISE/sw8/anaconda/anaconda3/lib/python3.9/site-packages/sklearn/cluster/__init__.py)

In [32]:
os.getcwd()

'/storage/work/fxs5261/soda501analysis'

In [33]:
os.makedirs("data_conditions", exist_ok=True)

In [34]:
#Lable games based on IVs

game_conditions = {
    "282070": {"MD": "Low",  "Agency": "High"},  # This War of Mine
    "1227530": {"MD": "Low",  "Agency": "High"}, # Partisans 1941
    "15390":  {"MD": "Low",  "Agency": "Low"},  # Brothers in Arms
    "50300":  {"MD": "Low",  "Agency": "Low"},  # Spec Ops
    "287700": {"MD": "High", "Agency": "High"}, # MGSV
    "460930": {"MD": "High", "Agency": "High"}, # Wildlands
    "400750": {"MD": "High", "Agency": "Low"},  # Gates of Hell
    "1029690":{"MD": "High", "Agency": "Low"}   # Sniper Elite 5
}

In [35]:
# Create a full dataset for all games

data_dir = "data"

dfs = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        app_id = file.replace(".csv", "")
        path = os.path.join(data_dir, file)

        df = pd.read_csv(path)

        if app_id in game_conditions:
            df["app_id"] = app_id
            df["MD"] = game_conditions[app_id]["MD"]
            df["Agency"] = game_conditions[app_id]["Agency"]

            dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True) 

print("Total samples:", len(full_df))

Total samples: 49815


## Condition-Based Subsets
We created multiple subsets of the data:

- By Moral Disengagement:
  - MD High
  - MD Low

- By Player Agency:
  - Agency High
  - Agency Low

- By combined conditions:
  - HH (High MD × High Agency)
  - HL (High MD × Low Agency)
  - LH (Low MD × High Agency)
  - LL (Low MD × Low Agency)

In [36]:
#Moral disengagement (High vs Low)
df_MD_H = full_df[full_df["MD"] == "High"]
df_MD_L = full_df[full_df["MD"] == "Low"]
print(len(df_MD_H), len(df_MD_L))

#Player agency (High vs Low)
df_Agency_H = full_df[full_df["Agency"] == "High"]
df_Agency_L = full_df[full_df["Agency"] == "Low"]
print(len(df_Agency_H), len(df_Agency_L))

# 2 x 2 conditions
df_HH = full_df[(full_df["MD"] == "High") & (full_df["Agency"] == "High")]
df_HL = full_df[(full_df["MD"] == "High") & (full_df["Agency"] == "Low")]
df_LH = full_df[(full_df["MD"] == "Low")  & (full_df["Agency"] == "High")]
df_LL = full_df[(full_df["MD"] == "Low")  & (full_df["Agency"] == "Low")]

print("HH:", len(df_HH))
print("HL:", len(df_HL))
print("LH:", len(df_LH))
print("LL:", len(df_LL))

19260 30555
34195 15620
HH: 14195
HL: 5065
LH: 20000
LL: 10555


In [37]:
# Save as csv files

output_dir = "data_conditions"

full_df.to_csv(os.path.join(output_dir, "full.csv"), index=False)

df_MD_H.to_csv(os.path.join(output_dir, "MD_H.csv"), index=False)
df_MD_L.to_csv(os.path.join(output_dir, "MD_L.csv"), index=False)

df_Agency_H.to_csv(os.path.join(output_dir, "Agency_H.csv"), index=False)
df_Agency_L.to_csv(os.path.join(output_dir, "Agency_L.csv"), index=False)

df_HH.to_csv(os.path.join(output_dir, "HH.csv"), index=False)
df_HL.to_csv(os.path.join(output_dir, "HL.csv"), index=False)
df_LH.to_csv(os.path.join(output_dir, "LH.csv"), index=False)
df_LL.to_csv(os.path.join(output_dir, "LL.csv"), index=False)

In [38]:
# Clean text using simple method (no spaCy needed)
full_df_clean = full_df.dropna(subset=["review_text"]).copy()
full_df_clean["clean_text"] = full_df_clean["review_text"].apply(simple_clean)

In [39]:
# Subset the Clean DF

df_MD_H_clean = full_df_clean[full_df_clean["MD"] == "High"]
df_MD_L_clean = full_df_clean[full_df_clean["MD"] == "Low"]
df_HH_clean = full_df_clean[(full_df_clean["MD"] == "High") & (full_df_clean["Agency"] == "High")]
df_HL_clean = full_df_clean[(full_df_clean["MD"] == "High") & (full_df_clean["Agency"] == "Low")]
df_LH_clean = full_df_clean[(full_df_clean["MD"] == "Low") & (full_df_clean["Agency"] == "High")]
df_LL_clean = full_df_clean[(full_df_clean["MD"] == "Low") & (full_df_clean["Agency"] == "Low")]

In [40]:
# Write the Clean DF to CSV

full_df_clean.to_csv(os.path.join(output_dir, "full_clean.csv"), index=False)
df_MD_H_clean.to_csv(os.path.join(output_dir, "MD_H_clean.csv"), index=False)
df_MD_L_clean.to_csv(os.path.join(output_dir, "MD_L_clean.csv"), index=False)
df_HH_clean.to_csv(os.path.join(output_dir, "HH_clean.csv"), index=False)
df_HL_clean.to_csv(os.path.join(output_dir, "HL_clean.csv"), index=False)
df_LH_clean.to_csv(os.path.join(output_dir, "LH_clean.csv"), index=False)
df_LL_clean.to_csv(os.path.join(output_dir, "LL_clean.csv"), index=False)

In [43]:
from bertopic import BERTopic

# Full corpus
docs = full_df_clean["clean_text"].tolist()
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

ModuleNotFoundError: No module named 'bertopic'